In [1]:
#  This notebook contains adding new 21 negative tweets to the process of LLM level1 filtering

In [ ]:
import openai
from openai import OpenAI
import pandas as pd
from typing import List
from datetime import datetime
from time import sleep
from os import path
#Set root path to the location of open oai_finetune_modules
import os
os.chdir("/llm_coding/")
import oai_finetune_modules as oaimd

In [5]:
def gen_response(submission_txt: str, q_txt: str,
                 filt_client: openai.Client, agent_type: str="neutral") -> str:
    """Function to generate a response given a text submission 
    and a single question to ask about the text.

    Args:
        submission_txt (str): The submission text about which questions need to be asked
        q_txt (str): The question to be asked about the text
        filt_client (openai.Client): The openAI client initialised to make API calls
        agent_type (str, optional): A string describing the style of the assistant's
                                    response. Currently only supports "Neutral". 
                                    Defaults to "neutral".

    Returns:
        str: The response given by the assistant to the text
    """
    if agent_type=="neutral":
        agent_type_str = ("You are a member of a population that is a high priority target for efforts related to reducing "
           "incidents of the Humman Immunodefiency Virus (For ex: Men who have sex with men).")

    submission_prompt = (f"I'm providing a text below, that I "
                         f"have some questions about:\n {submission_txt}")

    prompts_dicts = [{"role": "system", "content": agent_type_str},
                     {"role": "user", "content": submission_prompt},
                     {"role": "user", "content": q_txt}]

    response = filt_client.chat.completions.create(model="gpt-4o-mini-2024-07-18", 
                                                   messages=prompts_dicts)

    response_txt = response.choices[0].message.content

    return response_txt

In [6]:
def code_submission(submission_txt: str, submission_id: str,
                    q_df: pd.DataFrame, filt_client: openai.Client, 
                    agent_type: str="neutral") -> pd.DataFrame:
    """Function which codes a submission according to a list of provided
    questions. The list of provided questions should be provided as a 
    dataframe with two columns: one for the question itself and one 
    containing the question id. 

    Args:
        submission_txt (str): The submission text that needs to be coded
        submission_id (str): A unique ID for the submission that needs to be coded
        q_df (pd.DataFrame): Dataframe of questions according to which the 
                             submission needs to be coded
        filt_client (openai.Client): The openAI client initialised to make API calls
        agent_type (str, optional): A string describing the style of the agent's
                                    response. Defaults to "neutral".

    Returns:
        pd.DataFrame: A row of the dataframe, containing the responses provided
                      by the agent as a column
    """

    q_prefix = "Answer with a yes, no or unsure, followed by a reason in a couple of sentences. Separate the yes/no/unsure from the reason with a \"|\""
    result_dict = {}
    result_dict["sub_id"] = submission_id
    for ix in q_df.index:
        q_question = q_df["question_prompt"][ix]
        q_prompt = f"{q_prefix}_{q_question}" if q_df["qid"][ix] != "q_whic_sti" else q_question
        result_dict[q_df["qid"][ix]] = gen_response(submission_txt, 
                                                    q_prompt,
                                                    filt_client)

    result_row = pd.DataFrame(result_dict, index = [0])

    return result_row

In [ ]:
# 0. SETTING INPUT PATHS
data_dir = "/outputs"
train_path = path.join(data_dir, "openai_finetune_recode_apr22.csv")
neg_path = path.join(data_dir, "hiv_msgs_working_20241118.csv")
ctr_path = path.join(data_dir, "control_msgs.txt")
prompts_path = "gpt_prompts.csv"

In [ ]:
# 1. PREPARING THE INPUT DATASET
#train_df is created from train_sample and neg_df.
train_sample = pd.read_csv(train_path, usecols=["Content", "msg_cdn"])
# change the "msg_cdn" column name to "label" for clarity_ This columns shows human labels_ 
train_sample.rename(columns={'msg_cdn': 'label'}, inplace=True)

neg_df = pd.read_csv(neg_path, usecols=["Content", "neg_revisit"])

In [9]:
neg_df["neg_revisit"].value_counts()

neg_revisit
Yes    38
No     36
Name: count, dtype: int64

In [10]:
ctr_df = []
with open(ctr_path, "r") as ip_fp:
    for ip_l in ip_fp:
        ctr_df.append(ip_l)

neg_df["label"] = "negative messages_" + neg_df["neg_revisit"]
ctr_df = pd.DataFrame({"Content": ctr_df, "label": "non HIV"})

train_df = pd.concat([train_sample, neg_df, ctr_df]).reset_index()
train_df["sub_id"] = train_df["label"] + "_" + train_df["index"].astype(str)
train_df = train_df.drop(columns=["index"])

In [11]:
print(train_df['label'].value_counts())
print() 
print(f"Total number of tweets are: {len(train_df['label'])} tweets.")

label
non HIV                  53
negative messages_Yes    38
negative messages_No     36
actionable               24
random hiv               24
actionable vetted        24
Name: count, dtype: int64

Total number of tweets are: 199 tweets.


In [ ]:
# 2. SETTING UP THE OPEN AI CLIENT
q_prompts = pd.read_csv(prompts_path)
API_KEY = '...'
filt_client = OpenAI(api_key=oaimd.API_KEY)
filt_client.chat.completions.create(messages=[{"role": "user", "content": "say this is a test"}], model = "gpt-4o-mini-2024-07-18")

In [16]:
train_df.head(2)

,label,Content,neg_revisit,sub_id
0,actionable,@StormySturgeon At least a condom machine woul...,NaN,actionable_0
1,actionable,It's 2023 y'all--use a condom,NaN,actionable_1


In [ ]:
subs_df = train_df
result_df_list = []
start = datetime.now()
for i in subs_df.index:
    sub_id = subs_df["sub_id"][i]
    # try:
    result_df = code_submission(subs_df["Content"][i], 
                                subs_df["sub_id"][i], 
                                q_prompts, filt_client)
    result_df_list.append(result_df)
    print(f"Finished handling sub {sub_id}")

end = datetime.now()
runtime = end-start
print(f"The time to code the submissions was {runtime}")

Finished handling sub actionable_0


In [ ]:
result_df = pd.concat(result_df_list).reset_index()
result_df = result_df.merge(subs_df[["sub_id", "Content"]], on = "sub_id")

result_analysis_df = result_df

result_analysis_df["sub_id"] = result_analysis_df["sub_id"].str.replace("_yes", "-yes")
result_analysis_df["sub_id"] = result_analysis_df["sub_id"].str.replace("_no", "-no")
# result_analysis_df["sub_id"] = result_analysis_df["sub_id"].str.replace("_maybe", "-maybe")
# result_analysis_df["sub_id"] = result_analysis_df["sub_id"].str.replace("-maybe?", "-maybe")

result_analysis_df.loc[result_analysis_df.sub_id.str.contains("negative"), "sub_id"].unique()

result_analysis_df["label"] = result_analysis_df["sub_id"].str.split("_", expand=True)[0]

for q_id in q_prompts["qid"]:
    
    ans_colname = q_id.replace("q_", "") + "_ans"
    rsn_colname = q_id.replace("q_", "") + "_rsn"

    result_q_id = result_analysis_df[q_id].str.split("|", expand=True)
    result_analysis_df[ans_colname] = result_q_id[0]
    result_analysis_df[rsn_colname] = result_q_id[1]
    
result_df.to_csv("I_tst.csv")
result_analysis_df.to_csv("I_analysis_tst.csv", index=False)